<a href="https://colab.research.google.com/github/arihant7x/BePractical-training/blob/main/Day9_Lab_Building_with_LLM_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE API AUTHENTICATION
# Run this cell first. Installs the Google GenAI SDK, Pydantic, and dotenv.
# ==============================================================================
!pip install -q -U google-genai pydantic python-dotenv tabulate

import os
import sys
import time
import json
import random
import getpass
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Official Google GenAI SDK
from google import genai
from google.genai import types
from google.genai.errors import APIError

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion (Zero-Hardcoding Policy)
# ------------------------------------------------------------------------------
# Never hardcode API keys directly in scripts!
# In Google Colab, use the Secrets Manager (🔑 icon on the left panel) as 'GEMINI_API_KEY'.
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini API Client initialized successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
✅ Google Gemini API Client initialized successfully!


In [2]:
# ==============================================================================
# SECTION 1: API ARCHITECTURE, STATELESSNESS & SECURITY HYGIENE
# ==============================================================================
"""
1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:
   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.
   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.
   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.

2. THE STATELESSNESS MENTAL MODEL:
   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.
   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list
     of previous (User, Model) turns and pass the cumulative array on every subsequent call.

3. MESSAGE ROLES MAPPING:
   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐
   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │
   ├──────────────────────┼─────────────────────────┼───────────────────────────┤
   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │
   │ User Message         │ role: 'user'            │ role: 'user'              │
   │ Model Response       │ role: 'assistant'       │ role: 'model'             │
   └──────────────────────┴─────────────────────────┴───────────────────────────┘
"""

"\n1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:\n   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.\n   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.\n   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.\n\n2. THE STATELESSNESS MENTAL MODEL:\n   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.\n   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list\n     of previous (User, Model) turns and pass the cumulative array on every subsequent call.\n\n3. MESSAGE ROLES MAPPING:\n   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐\n   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │\n   ├──────────────────────┼─────────────────────────┼───────────────────────────┤\n   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │\

In [3]:
# ==============================================================================
# SECTION 2: PRE-FLIGHT TOKEN COUNTING & FINANCIAL COST ESTIMATION (UPDATED)
# ==============================================================================
"""
COST ESTIMATION BEST PRACTICE:
Count input tokens BEFORE invoking expensive generation calls to protect budget thresholds.
"""
import numpy as np
def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = "gemini-3.6-flash",
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """Calculates exact input tokens and estimates financial cost before calling the API."""
    # Count tokens using official Gemini Tokenizer
    token_resp = client.models.count_tokens(model=model_name, contents=text_prompt)
    input_tokens = token_resp.total_tokens

    # Official Rates per 1M tokens (USD)
    pricing = {
        "gemini-3.6-flash": {"in": 0.075, "out": 0.30},
        "gemini-1.5-pro":   {"in": 1.25,  "out": 5.00}
    }
    rate = pricing.get(model_name, pricing["gemini-3.6-flash"])

    est_cost = (input_tokens / 1e6 * rate["in"]) + (expected_output_tokens / 1e6 * rate["out"])

    return {
        "model": model_name,
        "input_tokens": input_tokens,
        "estimated_output_tokens": expected_output_tokens,
        "estimated_cost_usd": np.round(est_cost, 6),
        "cost_per_10k_calls": np.round(est_cost * 10000, 2)
    }

sample_payload = "Please summarize the last 10 quarterly financial filings of Apple, Microsoft, and Google."
# Updated to gemini-3.6-flash
estimate = preflight_cost_estimate(sample_payload, model_name="gemini-3.6-flash")
print("=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===")
for k, v in estimate.items():
    print(f"• {k:25s}: {v}")

=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===
• model                    : gemini-3.6-flash
• input_tokens             : 19
• estimated_output_tokens  : 500
• estimated_cost_usd       : 0.000151
• cost_per_10k_calls       : 1.51


In [4]:
# ==============================================================================
# SECTION 3: PRODUCTION RESILIENCE — EXPONENTIAL BACKOFF & RETRY LOOP
# ==============================================================================
"""
HANDLING API FAILURES IN PRODUCTION:
1. Rate Limits (HTTP 429 / ResourceExhausted): Hit requests-per-minute (RPM) ceiling.
2. Transient Server Errors (HTTP 500 / 503): Temporary Google Cloud infrastructure hiccup.
3. Network Timeouts: Connection dropped during streaming.

REMEDY: EXPONENTIAL BACKOFF WITH JITTER:
Wait time = (base_delay * 2^attempt) + random_jitter
Prevents "Thundering Herd" problem where all failed clients retry at the exact same millisecond.
"""

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """Wraps an API call in an exponential backoff retry loop with random jitter."""
    for attempt in range(max_retries):
        try:
            return api_call_func()
        except APIError as e:
            if attempt == max_retries - 1:
                print(f"❌ Max retries reached. Fatal API Error: {e}")
                raise e
            # Calculate backoff delay with jitter
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.1, 0.8)
            print(f"⚠️ Warning: Transient API Error ({e.code}). Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(delay)

In [5]:
# ==============================================================================
# SECTION 4: REUSABLE GEMINI WRAPPER & 3-TURN CHAT
# ==============================================================================
# Construct a reusable production function supporting:
# - Streaming responses (Low Time-To-First-Token)
# - System instructions
# - Dynamic temperature
# - Exponential backoff retry logic

def gemini_call(
    prompt: str,
    system_instruction: str = "You are a concise, helpful enterprise AI assistant.",
    temperature: float = 0.2,
    stream: bool = False,
    model: str = "gemini-3.6-flash"
) -> str:
    """Production-grade wrapper for Google Gemini API with error handling and streaming."""
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=800
    )

    if stream:
        def stream_call():
            full_text = []
            response_stream = client.models.generate_content_stream(
                model=model, contents=prompt, config=config
            )
            for chunk in response_stream:
                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_text.append(chunk.text)
            print() # Print final newline
            return "".join(full_text)

        return execute_with_exponential_backoff(stream_call)
    else:
        def standard_call():
            resp = client.models.generate_content(
                model=model, contents=prompt, config=config
            )
            return resp.text.strip()

        return execute_with_exponential_backoff(standard_call)

# ------------------------------------------------------------------------------
# 3-Turn Conversational Memory Loop Demonstration
# ------------------------------------------------------------------------------
print("=== MULTI-TURN CONVERSATION LOOP ===")

# Explicitly maintain stateless conversation history
conversation_history = []
system_persona = "You are a Senior PostgreSQL Database Administrator. Answer concisely in 2 sentences."

def send_chat_turn(user_message: str):
    print(f"\n👤 User: {user_message}")
    print("🤖 Assistant: ", end="")

    # 1. Append user message to history
    conversation_history.append({"role": "user", "parts": [{"text": user_message}]})

    # 2. Call Gemini passing full conversation history
    config = types.GenerateContentConfig(
        system_instruction=system_persona,
        temperature=0.0
    )
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=conversation_history,
        config=config
    )

    bot_reply = response.text.strip()
    print(bot_reply)

    # 3. Append model response to history to maintain context
    conversation_history.append({"role": "model", "parts": [{"text": bot_reply}]})

# Execute 3-Turn Dialogue (Demonstrating Context Memory)
send_chat_turn("What is the difference between a clustered and non-clustered index?")
send_chat_turn("Which one is faster for range queries on primary keys?") # Pronoun resolution!
send_chat_turn("Can a table have multiple of the faster one?")           # Contextual follow-up!

=== MULTI-TURN CONVERSATION LOOP ===

👤 User: What is the difference between a clustered and non-clustered index?
🤖 Assistant: A clustered index dictates the physical storage order of table data on disk so that the actual rows are sorted to match the index. Conversely, a non-clustered index creates a separate structure containing key values and row pointers, leaving the table's underlying physical layout unchanged.

👤 User: Which one is faster for range queries on primary keys?
🤖 Assistant: A clustered index is faster for range queries because the data rows are physically stored sequentially on disk, allowing the engine to perform efficient sequential reads. In contrast, a non-clustered index requires fetching rows scattered across different heap pages, resulting in expensive random I/O overhead for each matching record.

👤 User: Can a table have multiple of the faster one?
🤖 Assistant: No, a table can only have a single clustered index. This limitation exists because the physical data

In [7]:
# ==============================================================================
# SECTION 5: STUDENT LAB WORKSPACE (PORTFOLIO APPLICATION)
# ==============================================================================

# TASK 1: DEFINE PYDANTIC SCHEMA FOR STRUCTURED RESUME OPTIMIZATION
class ResumeBulletOptimization(BaseModel):
    original_bullet: str = Field(..., description="The raw bullet point provided by the user")
    xyz_formatted_bullet: str = Field(..., description="Rewritten using Google's XYZ formula: 'Accomplished [X], as measured by [Y], by doing [Z]'")
    impact_metric: str = Field(..., description="The quantifiable numeric KPI extracted or implied")
    action_verb: str = Field(..., description="The strong opening action verb used")
    seniority_score: int = Field(..., ge=1, le=10, description="Executive presence rating from 1 to 10")
    critique: str = Field(..., description="One sentence explaining what was improved")

# TASK 2: BUILD THE APPLICATION ENGINE
def optimize_resume_bullet(bullet: str, feedback: str = None) -> ResumeBulletOptimization:
    """Optimize a weak resume bullet into an executive XYZ-formatted one."""
    system_prompt = (
        "You are an expert executive career coach. Rewrite weak resume bullets "
        "using Google's XYZ formula: 'Accomplished [X], as measured by [Y], by doing [Z]'. "
        "If the user bullet lacks numbers, infer a realistic, plausible metric and note it in the critique."
    )
    if feedback:
        system_prompt += f"\nThe user requested this revision: {feedback}"

    config = types.GenerateContentConfig(
        system_instruction=system_prompt,
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=ResumeBulletOptimization,
    )

    def api_call():
        resp = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=f"Optimize this resume bullet: {bullet}",
            config=config,
        )
        return ResumeBulletOptimization.model_validate_json(resp.text)

    return execute_with_exponential_backoff(api_call)

# TASK 3: TEST APPLICATION ON REAL-WORLD WEAK BULLETS
result = optimize_resume_bullet(
    "Responsible for managing social media accounts and posting content regularly"
)
print(result.model_dump_json(indent=2))

revision = optimize_resume_bullet(
    "Responsible for managing social media accounts and posting content regularly",
    feedback="Make it sound more senior and use a revenue-related metric instead."
)
print(revision.model_dump_json(indent=2))

{
  "original_bullet": "Responsible for managing social media accounts and posting content regularly",
  "xyz_formatted_bullet": "Expanded total digital audience reach and grew cross-channel engagement by 45%, as measured by 25,000 net new organic followers over 6 months, by implementing a data-driven content strategy and optimizing audience targeting across key social platforms.",
  "impact_metric": "45% increase in engagement and 25,000 net new followers",
  "action_verb": "Expanded",
  "seniority_score": 8,
  "critique": "Transformed passive task-oriented language into a measurable strategic accomplishment using inferred engagement growth metrics."
}
{
  "original_bullet": "Responsible for managing social media accounts and posting content regularly",
  "xyz_formatted_bullet": "Spearheaded multi-channel social media strategy, generating $1.2M in attributed pipeline revenue, as measured by a 45% increase in marketing-driven sales conversions, by orchestrating targeted content campaig

In [8]:
# ==============================================================================
# SECTION 6: GIT REPOSITORY HYGIENE — CREATING .ENV AND .GITIGNORE
# ==============================================================================
"""
INSTRUCTIONS FOR PUSHING TO GITHUB SAFELY:

1. Create a `.env` file locally:
   GEMINI_API_KEY=your_actual_key_here

2. Create a `.gitignore` file in your project root containing:
   .env
   .env.local
   *.joblib
   __pycache__/
   .ipynb_checkpoints/

3. In your Python script (`app.py`), load the key cleanly via:
   from dotenv import load_dotenv
   load_dotenv()
   api_key = os.getenv("GEMINI_API_KEY")
"""

# Script to generate .gitignore locally in Colab
with open(".gitignore", "w") as f:
    f.write(".env\n.env.*\n*.joblib\n__pycache__/\n.ipynb_checkpoints/\n")

print("✅ '.gitignore' template created successfully!")

✅ '.gitignore' template created successfully!


In [11]:
with open(".env", "w") as f:
    f.write("GEMINI_API_KEY=your_actual_key_here")